# Barebones user-split buckets

This notebook only computes user membership buckets for train/val.

In [1]:
from pathlib import Path
import pandas as pd

# settings to not hide dataframe data in pandas. 
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / 'pyproject.toml').is_file():
            return d
    return here


PROJECT_ROOT = _repo_root()
PROCESSED = PROJECT_ROOT / 'data' / 'processed'

TRAIN_PARQUET = PROCESSED / 'steam_reviews_cleaned_english_train_norm.parquet'
VAL_PARQUET = PROCESSED / 'steam_reviews_cleaned_english_val_norm.parquet'
TEST_PARQUET = PROCESSED / 'steam_reviews_cleaned_english_test_norm.parquet'
USER_COL = 'author.steamid'

print('project_root:', PROJECT_ROOT)
print('processed_dir:', PROCESSED)
print('train exists:', TRAIN_PARQUET.is_file())
print('val exists:', VAL_PARQUET.is_file())
print('test exists:', TEST_PARQUET.is_file())


def load_users(path: Path) -> set[str]:
    if not path.is_file():
        return set()
    s = pd.read_parquet(path, columns=[USER_COL])[USER_COL].astype(str)
    return set(s.unique().tolist())


train_users = load_users(TRAIN_PARQUET)
val_users = load_users(VAL_PARQUET)
test_users = load_users(TEST_PARQUET)

# Universe includes test when available; otherwise train/val only.
universe_users = (train_users | val_users | test_users) if test_users else (train_users | val_users)

buckets = {
    'in_train_not_in_val': train_users - val_users,
    'in_train_in_val': train_users & val_users,
    'not_in_train_in_val': val_users - train_users,
    'not_in_train_not_in_val': universe_users - (train_users | val_users),
}

summary = pd.DataFrame([
    {
        'bucket': name,
        'n_users': len(users),
        'pct_of_universe': (len(users) / len(universe_users) * 100.0) if universe_users else 0.0,
    }
    for name, users in buckets.items()
]).sort_values('bucket').reset_index(drop=True)

print(f'train users: {len(train_users):,}')
print(f'val users: {len(val_users):,}')
print(f'test users: {len(test_users):,}')
print(f'universe users: {len(universe_users):,}')
display(summary)

sample = pd.DataFrame([
    {'bucket': name, 'sample_user_ids': sorted(list(users))[:5]}
    for name, users in buckets.items()
]).sort_values('bucket').reset_index(drop=True)
display(sample)


project_root: /home/ryanr/workspace/steam_recommendations
processed_dir: /home/ryanr/workspace/steam_recommendations/data/processed
train exists: True
val exists: True
test exists: True
train users: 4,010,019
val users: 1,579,462
test users: 1,580,391
universe users: 5,108,608


,bucket,n_users,pct_of_universe
0,in_train_in_val,1009920,19.768986
1,in_train_not_in_val,3000099,58.726350
2,not_in_train_in_val,569542,11.148673
3,not_in_train_not_in_val,529047,10.355991


,bucket,sample_user_ids
0,in_train_in_val,"[76561197960265778, 76561197960265781, 76561197960265858, 76561197960266146, 76561197960266375]"
1,in_train_not_in_val,"[76561197960265730, 76561197960265745, 76561197960265817, 76561197960265836, 76561197960265861]"
2,not_in_train_in_val,"[76561197960265747, 76561197960266109, 76561197960266531, 76561197960267113, 76561197960267807]"
3,not_in_train_not_in_val,"[76561197960265806, 76561197960265822, 76561197960265916, 76561197960267064, 76561197960267550]"


## Inspect full rows for Z users

Set **`N_TRAIN_ROWS` (X)**, **`N_VAL_ROWS` (Y)**, and **`Z`** (how many users). The next cell finds users in **train ∩ val** with exactly X train rows and Y val rows, takes the **first Z** from that list, and loads **all columns** into **one table**: first column **`split`** is `train` or `val`, then all parquet fields (use `USER_COL` to group by person).

In [2]:
# For each bucket, find users that appear more than once in their respective source parquet file.

def count_multirecord_users(parquet_path: Path, user_set: set[str]) -> int:
    """Count users from user_set with more than one record in the given parquet file."""
    if not parquet_path.is_file() or not user_set:
        return 0
    df = pd.read_parquet(parquet_path, columns=[USER_COL])
    # Only consider rows where the user is in the bucket's user_set
    filtered = df[df[USER_COL].astype(str).isin(user_set)]
    user_counts = filtered[USER_COL].astype(str).value_counts()
    return (user_counts > 1).sum()

multi_record_data = []
for bucket_name, users in buckets.items():
    if bucket_name.startswith('in_train'):
        source = TRAIN_PARQUET
    else:
        source = VAL_PARQUET
    multi_count = count_multirecord_users(source, users)
    multi_record_data.append({
        'bucket': bucket_name,
        'n_multi_record_users': multi_count,
        'pct_of_bucket': (multi_count / len(users) * 100.0) if users else 0.0,
    })

multi_df = pd.DataFrame(multi_record_data).sort_values('bucket').reset_index(drop=True)
print("Users with multiple records per bucket:")
display(multi_df)

Users with multiple records per bucket:


,bucket,n_multi_record_users,pct_of_bucket
0,in_train_in_val,462421,45.787884
1,in_train_not_in_val,435991,14.532554
2,not_in_train_in_val,20245,3.554611
3,not_in_train_not_in_val,0,0.000000


In [3]:
# For users that are in both train and val, count how many records they have in each dataset group.

# Find users present in both train and val
train_user_ids = set(train_users)
val_user_ids = set(val_users)
overlap_users = train_user_ids & val_user_ids

print(f'Users in both train and val: {len(overlap_users):,}')

def user_record_counts(parquet_path: Path, user_ids: set[str]) -> pd.Series:
    """Return a Series mapping user_id to their record count in the given parquet file."""
    if not parquet_path.is_file() or not user_ids:
        return pd.Series(dtype=int)
    df = pd.read_parquet(parquet_path, columns=[USER_COL])
    filtered = df[df[USER_COL].astype(str).isin(user_ids)]
    return filtered[USER_COL].astype(str).value_counts()

train_counts = user_record_counts(TRAIN_PARQUET, overlap_users)
val_counts = user_record_counts(VAL_PARQUET, overlap_users)

# Merge counts for each user present in both train and val
overlap_counts_df = pd.DataFrame({
    'user_id': list(overlap_users),
    'n_records_train': [train_counts.get(uid, 0) for uid in overlap_users],
    'n_records_val': [val_counts.get(uid, 0) for uid in overlap_users],
})

# Show value counts of n_records_train and n_records_val
print("Value counts for n_records_train:")
display(overlap_counts_df[['n_records_train','n_records_val']].value_counts().sort_index())

# print("Value counts for n_records_val:")
# display(overlap_counts_df['n_records_val'].value_counts().sort_index())

Users in both train and val: 1,009,920
Value counts for n_records_train:


n_records_train  n_records_val
1                1                547499
2                1                175692
3                1                 96844
4                1                 58287
5                1                 36775
6                1                 24262
7                1                 16971
8                1                 12271
9                1                  8919
10               1                  6799
11               1                  5079
12               1                  3836
13               1                  3076
14               1                  2396
15               1                  1941
16               1                  1557
17               1                  1221
18               1                  1000
19               1                   874
20               1                   731
21               1                   576
22               1                   497
23               1                   385
24               1        

In [4]:
# Run the first code cell above so TRAIN_PARQUET / VAL_PARQUET / USER_COL exist.
from pathlib import Path

import pandas as pd

N_TRAIN_ROWS = 3  # X — exact row count in train per user
N_VAL_ROWS = 1  # Y — exact row count in val per user
Z = 5  # take this many users from the matching list (first Z)

if "TRAIN_PARQUET" not in globals():

    def _repo_root() -> Path:
        here = Path.cwd().resolve()
        for d in [here, *here.parents]:
            if (d / "pyproject.toml").is_file():
                return d
        return here

    _root = _repo_root()
    _proc = _root / "data" / "processed"
    TRAIN_PARQUET = _proc / "steam_reviews_cleaned_english_train_norm.parquet"
    VAL_PARQUET = _proc / "steam_reviews_cleaned_english_val_norm.parquet"
    USER_COL = "author.steamid"

# --- row counts per user (single column reads) ---
_train_vc = pd.read_parquet(TRAIN_PARQUET, columns=[USER_COL])[USER_COL].astype(str).value_counts()
_val_vc = pd.read_parquet(VAL_PARQUET, columns=[USER_COL])[USER_COL].astype(str).value_counts()

_train_n = _train_vc.rename("n_train").to_frame()
_val_n = _val_vc.rename("n_val").to_frame()
_counts = _train_n.join(_val_n, how="inner").fillna(0).astype(int)

_mask = (_counts["n_train"] == N_TRAIN_ROWS) & (_counts["n_val"] == N_VAL_ROWS)
_candidates = _counts.loc[_mask].index.tolist()

if not _candidates:
    print(
        f"No user with exactly train={N_TRAIN_ROWS}, val={N_VAL_ROWS}. "
        "Try other X/Y or inspect _counts.describe()."
    )
    # show nearby counts for debugging
    _near = _counts[
        (_counts["n_train"].between(max(0, N_TRAIN_ROWS - 1), N_TRAIN_ROWS + 1))
        & (_counts["n_val"].between(max(0, N_VAL_ROWS - 1), N_VAL_ROWS + 1))
    ].head(20)
    display(_near)
else:
    z_take = min(int(Z), len(_candidates))
    picked_uids = [str(u) for u in _candidates[:z_take]]
    print(
        f"Picked {len(picked_uids)} user(s) (Z={Z}, available={len(_candidates)}) | "
        f"train rows/user={N_TRAIN_ROWS} | val rows/user={N_VAL_ROWS}"
    )
    print("user_ids:", picked_uids)

    def _read_user_rows(path: Path, uid: str) -> pd.DataFrame:
        try:
            return pd.read_parquet(path, filters=[(USER_COL, "==", uid)])
        except Exception:
            d = pd.read_parquet(path)
            return d[d[USER_COL].astype(str) == uid]

    train_parts = []
    val_parts = []
    for uid in picked_uids:
        train_parts.append(_read_user_rows(TRAIN_PARQUET, uid))
        val_parts.append(_read_user_rows(VAL_PARQUET, uid))

    df_train_all = pd.concat(train_parts, ignore_index=True)
    df_val_all = pd.concat(val_parts, ignore_index=True)

    print("\nRow counts check (per user):")
    display(
        pd.DataFrame(
            {
                "user_id": picked_uids,
                "n_train": [len(t) for t in train_parts],
                "n_val": [len(v) for v in val_parts],
            }
        )
    )

    _t = df_train_all.copy()
    _t.insert(0, "split", "train")
    _v = df_val_all.copy()
    _v.insert(0, "split", "val")
    df_all = pd.concat([_t, _v], ignore_index=True)

    print("\n--- TRAIN + VAL: full rows (combined; first column = split) ---")
    display(df_all.sort_values(['author.steamid','split']))
    print(
        f"rows — train: {len(df_train_all)} | val: {len(df_val_all)} | combined: {len(df_all)}"
    )


Picked 5 user(s) (Z=5, available=96844) | train rows/user=3 | val rows/user=1
user_ids: ['76561198120712786', '76561198874692720', '76561198151216181', '76561198847481270', '76561198124904381']

Row counts check (per user):


,user_id,n_train,n_val
0,76561198120712786,3,1
1,76561198874692720,3,1
2,76561198151216181,3,1
3,76561198847481270,3,1
4,76561198124904381,3,1



--- TRAIN + VAL: full rows (combined; first column = split) ---


,split,review_id,app_id,app_name,author.steamid,language,review,recommended,votes_helpful,author.num_games_owned,author.num_reviews,author.playtime_last_two_weeks,author.playtime_at_review,steam_purchase,received_for_free,written_during_early_access,timestamp_created,author.last_played,is_helpful,review_word_count,review_length_chars,review_age_seconds,_norm_votes_helpful,_norm_author__playtime_last_two_weeks,_norm_author__playtime_at_review,_norm_author__num_games_owned,_norm_author__num_reviews,_norm_review_word_count
0,train,84496579,292030,The Witcher 3: Wild Hunt,76561198120712786,english,"What can I say that hasn't been said before. Simply amazing sum up this game. If you like story rich, open world RPGs then buy the whole thing, expansions and all. In my book it ranks up there with Red Dead 2 as one of the finest games ever made. So much to do, an awesome story, great handling, and at this point in it's life it is a great value. What you get is so much more than what you pay for, and even having been out for years now the visuals still hold up. A true 5 out of 5 game.",1,0,97,12,0.0,23207.0,1,0,0,1610378165,1.557358e+09,0,103,489,1046926.0,0.000000,0.000000,10.052252,4.584967,2.564949,4.644391
1,train,72630326,346110,ARK: Survival Evolved,76561198120712786,english,"I have a love hate relationship with this game. At it's best it is a highly challenging survival/crafter which will keep you one your toes and paranoid about what's hiding behind the next rock. At it's worst this game can become a glitch ridden mess that will make you wonder why you keep it on your HD. \n\nSo to start off the dinos and maps look awesome. One of the best things in this game is exploring and finding all the hidden little nooks and details hidden within. The search for that perfect base location can take you hours if you aren't careful. There are so many cool features loaded into each map that sometimes the hardest thing you will do is pick a spot. The wildlife is always on the move and this keeps you on your toes, and lends a near constant sense of danger. \nOnce the main part of the vanilla game has worn thin you can turn to the large and active modding community. From maps, to QL improvments, to building assests the mods can really refresh this game.\nOn the done side the game is absolutely plagued by glitches. It's gotten better since I first purchased the game, but you will sometimes find yourself frustrated that a game this far into development is still the riddled with some of these problems. Watching as a bred and imprinted dino vanishes into a fold in the terrain as you dismount will just make you want to throw up your hands and walk away, and that is just the most common one I tend to come across. It sometimes feels like you are participating in the worlds longest beta test.\n\nI would probably suggest fans of this style of game to give the base game a try before dropping anything into expansions. I enjoy the swim, but newcomers are urged to dip a toe in first.",1,1,97,12,844.0,91237.0,1,0,0,1594660072,1.610652e+09,1,318,1707,16765019.0,0.693147,6.739337,11.421227,4.584967,2.564949,5.765191
2,train,72628470,440900,Conan Exiles,76561198120712786,english,"Great fun this game. Very pretty, and sets a great stage for you to play on. The building pieces snap about as well as most of these types of games, but look great. If you like building the DLCS are 100% worth it. Dungeon progression helps break up the grind, though I can't help but wish they were longer with a bit more challenge from the bosses. \nThralls are very powerful even post nerf patch, and pets could really use some TLC to make it a more difficult choice between the two. The followers can get buggy as all get out sometimes, leaving you frustrated while they stand there and stare while frost giants smash your head in. Predictable spawn areas, and no free enemy movement can make the game feel super easy a lot of the time. \nI don't play PvP so I can't comment on any of that. \nO

rows — train: 15 | val: 5 | combined: 20
